# Reproducible Code for: Neutral mutations promote unbounded adaptation after clonal interference.

### Salvador León Fernández, Miguel Ángel Fortuna Alcolado.

Here, we will merge the data of the most abundant organism every 1000 updates from `db_mut_load.csv` with `db_mut_supply.csv` to extract its fitness measurement. We also included an ancestor file because `db_mut_supply` only contains mutants and not the original sequences.

---

## Setup & Environment

In [2]:
library("data.table")
library("tidyverse")
library("scales")

## Ancestor file

In [3]:
db_fixed <- read_csv("../../DATA/db_fixed_mutations.csv",
                     col_types = cols("i", "i", "i", "i", "c", "d", "c", "i", "d", "d", "c", "i")) %>%
    mutate(sequence = str_remove_all(sequence, "\\*")) %>%
    arrange(desc(resource), rep_id, update) %>%
    mutate(length = nchar(sequence))
db_fixed %>% nrow
db_fixed %>% head(5)
db_fixed %>% tail(5)

[1] 26735

rep_id,founder_id,env_id,phenotype_id,resource,mut_rate,mut_type,update,fitness,relative_fitness,sequence,length
<int>,<int>,<int>,<int>,<chr>,<dbl>,<chr>,<int>,<dbl>,<dbl>,<chr>,<int>
1,1012,1659,1,NOT,0.001,NA,0,0.0731707,0.00000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,100
1,1012,1659,1,NOT,0.001,sub,1911,0.0731707,1.00000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzztpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,100
1,1012,1659,1,NOT,0.001,sub,8688,0.0757576,1.03535,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzztpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,100
1,1012,1659,1,NOT,0.001,sub,25860,0.0867052,1.14451,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzztpmpojwqyetaaopwaklmrsjlsazvuviaxgjudyjaodjfhx,100
1,1012,1659,1,NOT,0.001,ins,38954,0.1060070,1.22261,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzztpmpojwqyetaaopwaklmrsjlsazvuviaxvgjudyjaodjfhx,101


rep_id,founder_id,env_id,phenotype_id,resource,mut_rate,mut_type,update,fitness,relative_fitness,sequence,length
<int>,<int>,<int>,<int>,<chr>,<dbl>,<chr>,<int>,<dbl>,<dbl>,<chr>,<int>
600,1317,1787,1,NOT,1e-04,sub,199622,0.0707547,1.23585,zwksjcfrcuuaxqhcsqfezvysqancxvduveyyvtmvsgoqadwzozhucorjyqaymfotobgebohqdvzqdskbhfvucmogoslveciuahpbl,101
600,1317,1787,1,NOT,1e-04,sub,247016,0.0744417,1.05211,zwksjcfrcuuaxqhcsqfezvyslancxvduveyyvtmvsgoqadwzozhucorjyqaymfotobgebohqdvzqdskbhfvucmogoslveciuahpbl,101
600,1317,1787,1,NOT,1e-04,sub,254751,0.0771208,1.03599,zwksjcfrcuuaxqhcsqfezvyslaucxvduveyyvtmvsgoqadwzozhucorjyqaymfotobgebohqdvzqdskbhfvucmogoslveciuahpbl,101
600,1317,1787,1,NOT,1e-04,sub,449680,0.0806452,1.04570,zwksjcfrcuuaxqhcsqfezvyslaucxvduveyyvtcvsgoqadwzozhucorjyqaymfotobgebohqdvzqdskbhfvucmogoslveciuahpbl,101
600,1317,1787,1,NOT,1e-04,sub,489163,0.0821918,1.01918,zwksjcfrcuuaxqhcsqfezvyslaucxvduveyyvycvsgoqadwzozhucorjyqaymfotobgebohqdvzqdskbhfvucmogoslveciuahpbl,101


In [7]:
ancestors = db_fixed %>% filter(update == 0, mut_rate > 0.0005) %>% group_by(founder_id, rep_id, mut_rate, update, fitness, relative_fitness, sequence) %>% summarise(.groups = "drop")
ancestors %>% head
ancestors %>% tail

founder_id,rep_id,mut_rate,update,fitness,relative_fitness,sequence
<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<chr>
1012,1,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx
1012,21,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx
1012,41,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx
1012,61,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx
1012,81,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx
1012,101,0.001,0,0.0731707,0,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxgathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx


founder_id,rep_id,mut_rate,update,fitness,relative_fitness,sequence
<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<chr>
5642,282,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo
5642,302,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo
5642,322,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo
5642,342,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo
5642,362,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo
5642,382,0.01,0,0.0652174,0,sfnwezdjjudpncbtyctyujabihoftevwcmmszdnaskwdjvqwtqcrqugcuydzyvrqckxaftbyyyfndtjerooueyhaleufdzxqvgeo


In [9]:
write_csv(ancestors, file = "../../DATA/db_all_ancestors.csv")

## Extract most abundant organism's fitness values

In [10]:
cat("1. Loading mutational_load.csv and filtering dominant lineages...\n")
db_load <- fread("../../DATA/db_mut_load.csv")

dominant_lineages <- db_load %>%
  filter(abundance == max(abundance), .by = c(update, mu, rep_id, founder_id)) %>%
  rename(sample_update = update) %>%
  as.data.table()

rm(db_load)
gc()

cat("2. Loading the unified ancestor file...\n")
ancestors_db <- fread("../../DATA/db_all_ancestors.csv")

cat("3. Loading fitness_all_mutations.csv (essential columns only)...\n")
all_mutations_db <- fread(
  "../../DATA/db_mut_supply.csv",
  select = c("founder_id", "rep_id", "mut_rate", "update", "fitness_org_id", "relative_fitness", "seq")
)

cat("4. Merging data and filling missing values with ancestors...\n")

final_dataset <- dominant_lineages %>%
  left_join(
    all_mutations_db,
    by = c("founder_id" = "founder_id", 
           "rep_id" = "rep_id", 
           "mu" = "mut_rate", 
           "sequence" = "seq")
  ) %>%
  rename(origin_update_mut = update) %>%
  left_join(
    ancestors_db,
    by = c("founder_id" = "founder_id", 
           "rep_id" = "rep_id", 
           "mu" = "mut_rate", 
           "sequence" = "sequence"),
    suffix = c("_mut", "_anc")
  ) %>%
  mutate(
    fitness_org_id = coalesce(fitness_org_id, fitness),
    relative_fitness = coalesce(relative_fitness_mut, relative_fitness_anc),
    origin_update = coalesce(as.numeric(origin_update_mut), 0)
  ) %>%
  select(-fitness, -relative_fitness_mut, -relative_fitness_anc, -origin_update_mut) %>%
  filter(!is.na(fitness_org_id)) %>%
  arrange(founder_id, mu, rep_id, sample_update)

cat("5. Process complete. Releasing RAM...\n")
rm(all_mutations_db, ancestors_db, dominant_lineages)
gc()

head(final_dataset)

1. Loading mutational_load.csv and filtering dominant lineages...


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1139082,60.9,32727284,1747.9,31873598,1702.3
Vcells,21287326,162.5,555505764,4238.2,661000177,5043.1


2. Loading the unified ancestor file...
3. Loading fitness_all_mutations.csv (essential columns only)...
4. Merging data and filling missing values with ancestors...
5. Process complete. Releasing RAM...


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1141742,61.0,729455703,38957.2,445766993,23806.6
Vcells,273118580,2083.8,6512573519,49687.0,7015788050,53526.3


founder_id,env_id,resource,mu,rep_id,sample_update,sequence,abundance,fitness_org_id,update,relative_fitness,origin_update
<int>,<int>,<chr>,<dbl>,<int>,<int>,<chr>,<int>,<dbl>,<int>,<dbl>,<dbl>
1012,1659,NOT,1e-04,401,44000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,6025,0.0757576,NA,1.03535,39479
1012,1659,NOT,1e-04,401,45000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,8680,0.0757576,NA,1.03535,39479
1012,1659,NOT,1e-04,401,46000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,9622,0.0757576,NA,1.03535,39479
1012,1659,NOT,1e-04,401,47000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,9759,0.0757576,NA,1.03535,39479
1012,1659,NOT,1e-04,401,48000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,9805,0.0757576,NA,1.03535,39479
1012,1659,NOT,1e-04,401,49000,qmwuukrmwxqyqruiytjaalgcirofzhipufxpufkswxkathsvjtxkzzgpmpojwqyetaaopwaklmrsjlsazvuviixgjudyjaodjfhx,9656,0.0757576,NA,1.03535,39479


In [11]:
final_dataset = final_dataset %>% filter(mu > 0.0005)

In [13]:
fwrite(final_dataset, "../../DATA/db_joined_most_abundant_organism.csv")